In [2]:
import asyncio
import time
import pandas as pd
from playwright.async_api import async_playwright
from urllib.parse import urljoin, urlparse

In [3]:
async def get_all_routes(page, base_url):
    print(f"[-] Memulai scraping link dari: {base_url}")
    await page.goto(base_url, wait_until="domcontentloaded", timeout=20000)

    # Ambil atribut href dari semua elemen <a> di halaman
    locators = page.locator("a[href]")
    count = await locators.count()
    
    routes = set()
    for i in range(count):
        href = await locators.nth(i).get_attribute("href")
        if href:
            # Konversi link relatif menjadi absolut (misal: "/about" -> "https://example.com/about")
            abs_url = urljoin(base_url, href)
            # Hapus anchor/fragment (misal: #section1)
            clean_url = abs_url.split("#")[0]
            if clean_url.startswith("http"):
                routes.add(clean_url)
                
    print(f"[+] Ditemukan {len(routes)} link unik.")
    return routes

In [4]:
async def audit_single_route(context, url):
    """
    Tahap 2: Execution - Mengunjungi satu route dan mengukur performa/status
    """
    page = await context.new_page()
    status_code = None
    error_msg = None
    load_time_ms = 0
    status = "PENDING"

    # Listener untuk menangkap response code utama
    def handle_response(response):
        nonlocal status_code
        if response.url == url or response.url == f"{url}/":
            status_code = response.status

    page.on("response", handle_response)

    start_time = time.perf_counter()
    try:
        # Buka halaman
        response = await page.goto(url, wait_until="load", timeout=TIMEOUT_MS)
        end_time = time.perf_counter()
        
        load_time_ms = round((end_time - start_time) * 1000, 2)
        
        if response:
            status_code = response.status # Timpa status code dari listener jika response ada

        # Logika Penentuan PASS/FAIL QA
        if status_code and status_code >= 400:
            status = "FAIL"
            error_msg = f"HTTP Status {status_code}"
        elif status_code is None:
             status = "FAIL"
             error_msg = "No Response Received"
        else:
            status = "PASS"

    except Exception as e:
        end_time = time.perf_counter()
        load_time_ms = round((end_time - start_time) * 1000, 2)
        status = "FAIL"
        
        # Penanganan Error Spesifik
        env_str = str(e)
        if "Timeout" in env_str:
            status_code = "TIMEOUT"
            error_msg = f"Page load hang > {TIMEOUT_MS}ms"
        elif "net::ERR_NAME_NOT_RESOLVED" in env_str:
            status_code = "DNS_ERROR"
            error_msg = "Domain tidak ditemukan"
        elif "net::ERR_CONNECTION_REFUSED" in env_str:
            status_code = "CONN_REFUSED"
            error_msg = "Koneksi ditolak server"
        else:
            status_code = "ERROR"
            error_msg = env_str.split("\n")[0] # Ambil baris pertama error

    await page.close()
    
    # Return data untuk Pandas
    return {
        "URL": url,
        "Status QA": status,
        "HTTP Code": status_code,
        "Load Time (ms)": load_time_ms,
        "Error Details": error_msg if error_msg else "-"
    }

In [5]:
async def main():
    async with async_playwright() as p:
        # Gunakan Chromium (lebih cepat dari Firefox/WebKit untuk tugas ini)
        browser = await p.chromium.launch(headless=True)
        # Context terisolasi
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) QA-Audit-Bot/1.0"
        )
        
        # Dummy page untuk scraping halaman utama
        scrape_page = await context.new_page()
        
        # === TAHAP 1: SCRAPE ROUTES ===
        routes_to_audit = await get_all_routes(scrape_page, BASE_URL)
        await scrape_page.close()

        if not routes_to_audit:
            print("[!] Tidak ada route yang ditemukan atau gagal load halaman utama.")
            await browser.close()
            return

        # === TAHAP 2: AUDIT ROUTES (Sequential untuk akurasi load time) ===
        print(f"\n[-] Memulai Audit Kinerja pada {len(routes_to_audit)} halaman...")
        audit_results = []
        
        counter = 1
        total = len(routes_to_audit)
        
        for url in sorted(routes_to_audit):
            print(f"[{counter}/{total}] Menguji: {url}")
            result = await audit_single_route(context, url)
            audit_results.append(result)
            
            # Print hasil singkat ke terminal
            print(f"    -> Hasil: {result['Status QA']} | Code: {result['HTTP Code']} | Time: {result['Load Time (ms)']}ms")
            counter += 1

        await browser.close()

        # === TAHAP 3: REPORTING DENGAN PANDAS ===
        print("\n[-] Menyusun Laporan...")
        df = pd.DataFrame(audit_results)

        # Mengurutkan berdasarkan Load Time tertinggi (opsional, untuk QA finding)
        df_sorted = df.sort_values(by="Load Time (ms)", ascending=False)

        # Export ke CSV
        filename = "qa_ui_route_audit.csv"
        df_sorted.to_csv(filename, index=False)
        print(f"[+] Laporan QA berhasil disimpan di: {filename}")

        # Tampilkan ringkasan di terminal
        total_fail = len(df[df["Status QA"] == "FAIL"])
        print("\n=== RINGKASAN AUDIT ===")
        print(f"Total Pages Scraped & Tested : {len(df)}")
        print(f"Passed                       : {len(df[df['Status QA'] == 'PASS'])}")
        print(f"Failed (Error/Slow)          : {total_fail}")
        if total_fail > 0:
            print("\n--- Daftar Halaman Gagal ---")
            print(df[df["Status QA"] == "FAIL"][["URL", "HTTP Code", "Error Details"]].to_string(index=False))


In [6]:
# --- KONFIGURASI ---
BASE_URL = "https://uad.ac.id/en/ahmad-dahlan-university/"  # GANTI DENGAN URL TARGET ANDA
TIMEOUT_MS = 20000  # 20 detik timeout per halaman

# Langsung panggil dengan await di sel Jupyter:
await main()

[-] Memulai scraping link dari: https://uad.ac.id/en/ahmad-dahlan-university/
[+] Ditemukan 46 link unik.

[-] Memulai Audit Kinerja pada 46 halaman...
[1/46] Menguji: http://bimawa.uad.ac.id/
    -> Hasil: PASS | Code: 200 | Time: 16596.49ms
[2/46] Menguji: http://bimawa.uad.ac.id/beasiswa/
    -> Hasil: PASS | Code: 200 | Time: 2427.08ms
[3/46] Menguji: http://cdc.uad.ac.id/
    -> Hasil: PASS | Code: 200 | Time: 5470.39ms
[4/46] Menguji: http://conference.uad.ac.id/
    -> Hasil: FAIL | Code: DNS_ERROR | Time: 49.64ms
[5/46] Menguji: http://digilib.uad.ac.id/
    -> Hasil: FAIL | Code: DNS_ERROR | Time: 46.25ms
[6/46] Menguji: http://elearning.uad.ac.id/
    -> Hasil: PASS | Code: 200 | Time: 3106.95ms
[7/46] Menguji: http://eprints.uad.ac.id/
    -> Hasil: FAIL | Code: TIMEOUT | Time: 20006.45ms
[8/46] Menguji: http://islamiccenter.uad.ac.id/
    -> Hasil: PASS | Code: 200 | Time: 4636.81ms
[9/46] Menguji: http://journal.uad.ac.id/
    -> Hasil: FAIL | Code: 403 | Time: 631.85ms
[1